In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_action_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LegalActionID String, SourceSystem String, PortfolioKey String, PortfolioName String, IngestionTimestamp timestamp,"
        #     "BatchId String, LegalActionKey String, LegalActionCode String, ExposureKey String, PartyKey String,"
        #     "CDI String, AgreementKey String,CaseCode string, CaseCodePrimary string, LegalActionCreatedDate timestamp, LegalActionCreatedBy String,"
        #     "LegalActionDate timestamp, LegalActionTypeCode String,LegalActionType String, LegalActionStatusCode String,"
        #     "LegalActionStatus String, LegalActionStepKey String, LegalActionStepTypeCode String,"
        #     "LegalActionStepType String, LegalActionStepStartDate timestamp,LegalActionStepEndDate timestamp,"
        #     "LegalTypeCode string,LegalType String, CompletionResultCode String, CompletionResultDate timestamp, CompletionResult String,"
        #     "LawyerPartyKey String,LawyerPartyName string, BankAccount string, Comments String, BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalAction"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qc",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== activity ==================

        activity_extra_filters = None

        if activity_extra_filters is not None:
            activity_extra_filters.append(date_filter)
            activity_final_filters = " AND ".join(activity_extra_filters)
        else:
            activity_final_filters = date_filter

        activity_selected_fields = [
            "cac_code",
            "cac_created",
            "vend_code",
            "cac_created_by",
            "cac_date",
            "sort_code",
            "extc_code",
            "comments",
            "case_code",
            "businessdate",
        ]

        activity = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_activity",  # check name
            activity_final_filters,
            activity_selected_fields,
        )
        # ================== activities_vw ==================

        activities_vw_extra_filters = None

        if activities_vw_extra_filters is not None:
            activities_vw_extra_filters.append(date_filter)
            activities_vw_final_filters = " AND ".join(activities_vw_extra_filters)
        else:
            activities_vw_final_filters = date_filter

        activities_vw_selected_fields = [
            "ACTV_INTCODE",
            "ACTV_IS_LEGAL",
            "ACTV_NAME",
        ]

        activities_vw = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_activities_vw",  # check name
            activities_vw_final_filters,
            activities_vw_selected_fields,
        )
        # ================== case_master ==================

        case_master_extra_filters = None

        if case_master_extra_filters is not None:
            case_master_extra_filters.append(date_filter)
            case_master_final_filters = " AND ".join(case_master_extra_filters)
        else:
            case_master_final_filters = date_filter

        case_master_selected_fields = [
            "vend_code",
            "old_case_code",
            "account_code",
            "cust_code",
            "case_code_primary",
            "case_code",
            "contract_number",
        ]

        case_master = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_case_master",  # check name
            case_master_final_filters,
            case_master_selected_fields,
        )
        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "cdi",
            "cust_code",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer",  # check name
            customer_final_filters,
            customer_selected_fields,
        )
        # ================== qc_external_companies ==================

        external_companies_extra_filters = None

        if external_companies_extra_filters is not None:
            external_companies_extra_filters.append(date_filter)
            external_companies_final_filters = " AND ".join(
                external_companies_extra_filters
            )
        else:
            external_companies_final_filters = date_filter

        external_companies_selected_fields = [
            "extc_code",
            "extc_name",
        ]

        external_companies = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_external_companies",  # check name
            external_companies_final_filters,
            external_companies_selected_fields,
        )
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "Prefix", "vendor_code_qualco"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== Join ==================

        activity_activities_vw_join_condition = (
            activity["sort_code"] == activities_vw["ACTV_INTCODE"]
        )

        activity_case_master_join_condition = (
            activity["case_code"] == case_master["case_code"]
        )

        case_master_customer_join_condition = (
            customer["cust_code"] == case_master["cust_code"]
        )

        activity_external_companies_join_condition = (
            activity["extc_code"] == external_companies["extc_code"]
        )

        activity_portfolio_join_condition = (
            activity["vend_code"] == portfolio["vendor_code_qualco"]
        )

        target_table_qc = (
            activity.join(
                activities_vw,
                activity_activities_vw_join_condition,
                "inner",
            )
            .join(
                case_master,
                activity_case_master_join_condition,
                "inner",
            )
            .join(
                customer,
                case_master_customer_join_condition,
                "inner",
            )
            .join(
                external_companies,
                activity_external_companies_join_condition,
                "left",
            )
            .join(
                portfolio,
                activity_portfolio_join_condition,
                "left",
            )
        ).where(activities_vw["ACTV_IS_LEGAL"] == 1)

        # HashKeys
        target_table_qc_LegalActionID_primary_key_fields = [
            activity.cac_code,
            lit("qc"),
        ]

        target_table_qc = target_table_qc.withColumn(
            "LegalActionID",
            sha2(
                concat_ws(";", *target_table_qc_LegalActionID_primary_key_fields), 256
            ),
        ).distinct()

        # ================== Selection of Final Columns ==================

        result_df_qualco = target_table_qc.select(
            target_table_qc.LegalActionID,
            lit("qc").alias("SourceSystem"),
            activity.vend_code.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            target_table_qc.cac_code.cast("string").alias("LegalActionKey"),
            target_table_qc.cac_code.cast("string").alias("LegalActionCode"),
            target_table_qc.old_case_code.cast("string").alias("ExposureKey"),
            case_master.cust_code.cast("string").alias("PartyKey"),
            customer.cdi.cast("String").alias("CDI"),
            case_master.contract_number.cast("String").alias("AgreementKey"),
            case_master.case_code.cast("String").alias("CaseCode"),
            target_table_qc.case_code_primary.cast("String").alias("CaseCodePrimary"),
            target_table_qc.cac_created.alias("LegalActionCreatedDate"),
            target_table_qc.cac_created_by.alias("LegalActionCreatedBy"),
            target_table_qc.cac_date.cast("timestamp").alias("LegalActionDate"),
            target_table_qc.sort_code.alias("LegalActionTypeCode"),
            activities_vw.ACTV_NAME.cast("string").alias("LegalActionType"),
            lit(None).alias("LegalActionStatusCode"),
            lit(None).alias("LegalActionStatus"),
            lit(None).alias("LegalActionStepKey"),
            lit(None).alias("LegalActionStepTypeCode"),
            lit(None).alias("LegalActionStepType"),
            lit(None).cast("timestamp").alias("LegalActionStepStartDate"),
            lit(None).cast("timestamp").alias("LegalActionStepEndDate"),
            lit(None).alias("LegalTypeCode"),
            lit(None).alias("LegalType"),
            lit(None).alias("CompletionResultCode"),
            lit(None).cast("timestamp").alias("CompletionResultDate"),
            lit(None).alias("CompletionResult"),
            activity.extc_code.cast("string").alias("LawyerPartyKey"),
            external_companies.extc_name.cast("string").alias("LawyerPartyName"),
            lit(None).cast("String").alias("BankAccount"),
            target_table_qc.comments.alias("Comments"),
            activity.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df_qualco.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_legal_action_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)